In [1]:
!pip install google-generativeai pillow langchain-chroma

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 9.9 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 3.8/3.8 MB 25.4 MB/s  0:00:00
   ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
   ---------------------------------------- 15.3/15.3 MB 74.4 MB/s  0:00:00

   ----------------------------------------  0/15 [uritemplate]
   -- -------------------------------------  1/15 [pyparsing]
   -- -------------------------------------  1/15 [pyparsing]
   ----- ----------------------------------  2/15 [pyasn1]
   ----- ------

In [ ]:
PDF_PATH = file:///C:/Users/laksh/Downloads/Environment_Report.pdf

In [3]:
import ollama
def ask_llm(prompt):
    response = ollama.chat(
        model = "llama3:8b",
        messages = [{
            "role" : "user",
            "content" : prompt
        }],
        stream = true
    )
    for res in response:
        print(res['messages']['content'],end="",flush = True)

In [4]:
import requests

def generate_response(prompt):

    response = requests.post(
        "https://myai.ganiisunkara.workers.dev",
        headers={
            "Authorization": "Bearer 12345678",
            "Content-Type": "application/json"
        },
        json={
            "prompt": prompt
        }
    )
    data = response.json()
    print(data['response'])
    return data['response']

In [7]:
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption
)
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True
pipeline_options.generate_page_images = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

result = converter.convert("C:/Users/laksh/Downloads/Environment_Report.pdf")

doc = result.document

[INFO] 2026-06-17 21:27:11,379 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-17 21:27:11,393 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\laksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-17 21:27:11,394 [RapidOCR] main.py:65: Using C:\Users\laksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-17 21:27:12,190 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-17 21:27:12,216 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\laksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-17 21:27:12,220 [RapidOCR] main.py:65: Using C:\Users\laksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-17 21:27:13,244 [RapidOCR] base

In [8]:
print(len(doc.pictures))

2


In [9]:
from docling.chunking import HybridChunker
chunker = HybridChunker()
chunks = list(chunker.chunk(dl_doc=doc))

In [18]:
!pip install google-generativeai

In [19]:
chunks


[DocChunk(text='The environment encompasses all living and non-living things around us, including forests, rivers, mountains, air, and wildlife. Healthy ecosystems provide clean water, fertile soil, and stable climates that support human life. However, rapid urbanization, pollution, and deforestation have placed immense pressure on natural resources. Protecting the environment is essential for ensuring a sustainable future for present and future generations.', meta=DocMeta(schema_name='docling_core.transforms.chunker.DocMeta', version='1.0.0', doc_items=[DocItem(self_ref='#/texts/2', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TEXT: 'text'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=78.0, t=487.06977050781245, r=508.16999999999985, b=429.81977050781245, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 446))], source=[], comments=[])], headings=['Environment: Our Shared Responsibility'],

In [10]:
import google.generativeai as genai

genai.configure(api_key="AQ.Ab8RN6J_jSy4CoaOkeZBuuR_ReKLYX3Y6WRjg38Pj-YkHKAUCg")

model = genai.GenerativeModel("gemini-2.5-flash")


C:\Users\laksh\AppData\Local\Temp\ipykernel_23084\2561677232.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [11]:


def summarize_chunk(chunk):
    TEXT_PROMPT = f"""
Summarize the following chunk.
Preserve important names, numbers and facts.
Return only the summary.

Chunk:
{chunk}
"""
    return generate_response(TEXT_PROMPT)

In [12]:
IMAGE_PROMPT = """
Generate a comprehensive semantic description of this image for use in a multimodal retrieval system.

Describe all visible content including:
- Titles and headings
- Objects and entities
- Text and numbers
- Tables and charts
- Axes, legends and units
- Diagrams and flowcharts
- Relationships between components
- Colors and layout when important
- Statistics and measurements
- Technical terminology
- Key conclusions and insights

Preserve names, dates, values, and identifiers exactly.

Produce a detailed, self-contained description so that a user can answer questions about the image without directly seeing it.

Return only the description.
"""

In [13]:
def summarize_image(image):

    response = model.generate_content(
        [
            IMAGE_PROMPT,
            image
        ]
    )

    print(response.text)

    return response.text

In [14]:
documents = []
from langchain_core.documents import Document


In [15]:
from langchain_core.documents import Document

documents = []

for chunk in chunks:
    print("chunked")
    summary = summarize_chunk(str(chunk.text))

    documents.append(
        Document(
            page_content=summary,
            metadata={
                "type": "text"
            }
        )
    )

chunked
The environment includes living and non-living things like forests, rivers, and wildlife, and healthy ecosystems support human life by providing clean water, fertile soil, and stable climates. However, rapid urbanization, pollution, and deforestation threaten natural resources, making environmental protection crucial for a sustainable future.
chunked
Climate change, biodiversity loss, plastic pollution, and excessive resource consumption are major environmental issues, with rising global temperatures causing extreme weather events and habitat destruction threatening species, requiring actions like reducing emissions, conserving ecosystems, and adopting renewable energy.
chunked
Individuals can contribute to environmental conservation through daily actions like conserving water and electricity, minimizing waste, and recycling, making a significant collective impact and supporting sustainability for future generations.


In [16]:
# Add images
for picture in doc.pictures:

    image = picture.get_image(doc)

    if image is None:
        continue

    summary = summarize_image(image)

    documents.append(
        Document(
            page_content=summary,
            metadata={
                "type": "image"
            }
        )
    )

This image is a minimalist, stylized graphic depicting a single tree against a landscape background, accompanied by a textual message. The overall color palette is dominated by light blue, dark green, and brown, with black text.

The image is divided horizontally into two main background areas:
1.  **Upper Background:** The top approximately two-thirds of the image is a solid light blue color, representing the sky.
2.  **Lower Background:** The bottom approximately one-third of the image is a solid dark green color, representing grass or ground. The boundary between the blue and green is a straight horizontal line.

A single, highly simplified tree is positioned centrally in the image, overlapping both the blue sky and green ground areas.
*   **Tree Foliage:** The canopy of the tree is represented by a large, solid dark green circle, occupying the upper-middle section of the image and extending into the blue sky area.
*   **Tree Trunk:** The trunk of the tree is a solid brown rectangle

In [17]:
len(documents)

5

In [18]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="mixedbread-ai/mxbai-embed-large-v1",
)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5227.02it/s]


In [19]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name="multimodal_rag",
    persist_directory="./chroma_mragdb",
    embedding_function=embeddings
)

In [20]:
vectorstore._collection.count()

4

In [21]:
vectorstore.add_documents(documents)

['501f501a-d33e-4886-98fb-315235fe874b',
 'ed83b742-a586-4a16-83d5-04a7cd22eeaa',
 '4d51fddf-aa1f-48d7-a30c-c3d8fe9ac854',
 '7fe6a79b-2aed-4464-baff-15a892f8a209',
 'bb0538dd-2338-487d-9a04-b75a3e40c40d']

In [22]:
vectorstore._collection.count()

9

In [23]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":20,
        "fetch_k":50
    }
)


In [25]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-large",
)


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2322.48it/s]


In [26]:
def retrieve(query, top_k=5):

    docs = retriever.invoke(query)

    pairs = [
        [query, doc.page_content]
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )

    return [
        doc
        for score, doc in ranked_docs[:top_k]
    ]

In [27]:
def build_context(query):

    retrieved_docs = retrieve(query)

    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    return context

In [28]:
import requests

def generate_response(prompt):

    response = requests.post(
        "https://text-generation.lakshmiparvathikosuru.workers.dev/",
        headers={
            "Authorization": "Bearer 87654321",
            "Content-Type": "application/json"
        },
        json={
            "prompt": prompt
        }
    )

    return response.json()

In [37]:
def build_context(query, k=3):
    docs = vectorstore.similarity_search(query, k=k)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    return context

In [38]:
def ask(query):

    context = build_context(query)
    print(f"The context retrived is:{context}")

    prompt = f"""
You are a helpful assistant.

Answer the question using ONLY the context below.

If the answer is not present, say:
'I could not find the answer in the provided context.'

Context:
{context}

Question:
{query}

Answer:
"""

    return generate_response(prompt)

In [ ]:
while True:
    inp = input("Enter your query : ")
    answer = ask(inp)
    print(answer)

In [40]:
from transformers import AutoTokenizer
from docling.chunking import HybridChunker

tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)



In [41]:
chunker = HybridChunker(
    tokenizer=tokenizer,
    max_tokens=1024
)

chunks = list(chunker.chunk(dl_doc=doc))

In [42]:
len(chunks)

3

In [43]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=600
)

docs = []

for chunk in chunks:
    docs.extend(
        splitter.create_documents(
            [chunk.text]
        )
    )

In [44]:
len(docs)

3

In [46]:
docs[1]

Document(metadata={}, page_content='Climate change, biodiversity loss, plastic pollution, and excessive consumption of resources are among the most pressing environmental issues today. Rising global temperatures contribute to extreme weather events, while habitat destruction threatens countless plant and animal species. Reducing emissions, conserving ecosystems, and adopting renewable energy sources are crucial steps toward addressing these challenges.')